In [15]:
import make_uv_samples as makeuv
import xarray as xr
from pathlib import Path
import numpy as np

In [2]:
TRUE_W = "/glade/work/iranjan/fast-osse/fastosse-run1/true_w.nc"
true_w = xr.open_dataset(TRUE_W)

In [3]:
print(true_w.time.values[:3])

['2015-01-02T12:00:00.000000000' '2015-01-09T12:00:00.000000000'
 '2015-01-16T12:00:00.000000000']


In [4]:
module_dir = Path("/glade/work/iranjan/tpose24-osse/")
sys.path.append(str(module_dir))

import osse_tools as ost

In [5]:
out_path = "/glade/derecho/scratch/iranjan/weekly_eep_osse"
joined_obs_seq = makeuv.join_obs_seq_outputs(out_path + "/outputs")

Found 25 obs_seq.out files
Total obs in joined sequence: 9250


In [6]:
joined_obs_seq = makeuv.join_obs_seq_outputs(out_path + "/outputs")
uv_samples = makeuv.reshape_obs_to_uv_samples(joined_obs_seq.df)

est_w = ost.compute_w_planefit(uv_samples, extrapolate_to_surface=False)

Found 25 obs_seq.out files
Total obs in joined sequence: 9250
Inferred cadence: 7 days 00:00:00. Snapped timestamps to this grid (max shift applied: 0 days 00:00:00).
Found 5 distinct glider positions.


In [7]:
print(est_w['w_est'].time.values[:3])

['2015-01-02T12:00:00.000000000' '2015-01-09T12:00:00.000000000'
 '2015-01-16T12:00:00.000000000']


In [26]:
w_diff = xr.Dataset({'w_diff': est_w['w_est'] - true_w['true_w_mean']})

In [27]:
print(w_diff)

<xarray.Dataset> Size: 8kB
Dimensions:  (time: 25, depth: 38)
Coordinates:
  * time     (time) datetime64[ns] 200B 2015-01-02T12:00:00 ... 2015-06-19T12...
  * depth    (depth) float64 304B -7.0 -9.0 -11.0 -13.0 ... -77.0 -79.0 -81.0
Data variables:
    w_diff   (time, depth) float64 8kB 1.161e-05 1.371e-05 ... 2.605e-05


In [28]:
print(true_w)

<xarray.Dataset> Size: 8kB
Dimensions:      (depth: 38, time: 25)
Coordinates:
  * depth        (depth) float64 304B -7.0 -9.0 -11.0 ... -77.0 -79.0 -81.0
  * time         (time) datetime64[ns] 200B 2015-01-02T12:00:00 ... 2015-06-1...
Data variables:
    true_w_mean  (time, depth) float64 8kB ...


In [29]:
print(est_w)

<xarray.Dataset> Size: 16kB
Dimensions:    (time: 25, depth: 38, obs_depth: 37)
Coordinates:
  * time       (time) datetime64[ns] 200B 2015-01-02T12:00:00 ... 2015-06-19T...
  * depth      (depth) float64 304B -7.0 -9.0 -11.0 -13.0 ... -77.0 -79.0 -81.0
  * obs_depth  (obs_depth) float64 296B -8.0 -10.0 -12.0 ... -76.0 -78.0 -80.0
Data variables:
    w_est      (time, depth) float64 8kB 0.0 -7.339e-07 ... 3.179e-05 3.35e-05
    div        (time, obs_depth) float64 7kB -3.67e-07 2.247e-07 ... 8.554e-07


In [31]:
np.linalg.norm(w_diff['w_diff'].values)

np.float64(0.0007089622661378881)